<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed;line-height:2;font-family:Tahoma,"IRANSans","Vazirmatn","Segoe UI",sans-serif;">

<div align="center">

## المپیاد هوش‌مصنوعی ایران

### آزمون مرحله‌ی سوم

**تابستان ۱۴۰۵**

</div>

---

**نام:** ..............................................

**نام خانوادگی:** ..............................................

---


</div>

<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">

# 📓 پیش‌بینی توان خروجی نیروگاه صنعتی

## 1️⃣ شرح مسئله

یک مجتمع نیروگاهی برای بهینه‌سازی مصرف سوخت، نیاز دارد **توان خروجی واقعی توربین‌ها (Power Output بر حسب Megawatt)** را به صورت لحظه‌ای پیش‌بینی کند.



سیستم کنترل نیروگاه دارای یک **محدودیت سخت‌افزاری در پردازنده مرکزی** است:



> پردازنده مرکزی کنترلر فقط توانایی محاسبه ضرب داخلی یک بردار وزن در بردار ویژگی‌ها را در مقیاس میکروثانیه دارد. بنابراین، **تیم مهندسی اجازه استفاده از مدل‌های غیرخطی (مانند شبکه‌های عصبی عمیق یا درخت‌های تصمیم) در خط تولید را ندارد** و مدل نهایی باید حتماً یک **مدل خطی روی فضای ویژگی‌ها(features)** باشد.
>
>
>

با این حال، این محدودیت به معنای ساده‌بودن مسئله نیست. داده‌های دریافتی از پایگاه داده عملیاتی خام، حاوی اختلالات سنسوری، نویزهای صنعتی و فرمت‌های ناهمخوان هستند. از سوی دیگر، رفتار فیزیکی توربین به شدت غیرخطی است.



پرسش اصلی این است:



> **چگونه می‌توان با کشف روابط پنهان در داده‌ها و طراحی یک فضای ویژگی جدید، یک مدل خطی ساده را قادر ساخت تا الگوهای پیچیده و غیرخطی سیستم را با دقت بالا پیش‌بینی کند؟**
>
>

در بخش‌های بعدی، برای ثبت کف مهارت فنی، پیاده‌سازی مدل خطی پایه از صفر الزامی است؛ اما پس از آن، فضای توسعه کاملاً باز است.


⚠️ **این محدودیت (مدل نهایی خطی) یک قید سخت و الزامی مسابقه است.** از آنجا که تحویل نهایی شما تنها یک فایل CSV است (بخش ۵)، سرور داوری هیچ کدی از شما اجرا یا بررسی نمی‌کند؛ اما نوت‌بوک شما پس از پایان مسابقه به‌صورت دستی بررسی می‌شود تا از رعایت این قید اطمینان حاصل شود. نوت‌بوک خود را دقیقاً به همان شکلی که مدل نهایی را تولید کرده، نگه دارید.
</div>

<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">

## 2️⃣ شناسنامه داده‌ها

دو فایل در اختیار شما قرار گرفته است: `train_data.csv` (داده‌ی آموزش، شامل ستون هدف `power_output`) و `test_queries.csv` (داده‌ی آزمون، **بدون** ستون `power_output`، با یک ستون اضافه‌ی `id`). برچسب‌های واقعی داده‌ی آزمون (`test_labels.csv`) نزد سرور داوری محفوظ است و هرگز در اختیار شما قرار نمی‌گیرد -- بخش ۵ و بخش ۱۱ توضیح می‌دهند چگونه با استفاده از `test_queries.csv` فایل تحویلی خود را می‌سازید.

فیلدهای موجود در داده به شرح زیر است:

| **نام فیلد** | **واحد / نوع** | **شرح فیلد** |
| --- | --- | --- |
| `temp` | درجه سانتی‌گراد (°C) | دمای ورودی محفظه احتراق/توربین |
| `ambient_temp` | درجه سانتی‌گراد (°C) | دمای هوای محیط در محل نیروگاه |
| `pressure` | بار (Bar) | فشار سیال عامل ورودی به توربین |
| `rpm` | دور بر دقیقه (RPM) | سرعت چرخش محور اصلی توربین |
| `hours_active` | ساعت (Hours) | مدت زمان کارکرد مداوم سیستم از آخرین سرویس/اورهال |
| `vibration` | میلی‌متر بر ثانیه (mm/s) | میزان لرزش بدنه توربین (سیگنال پایش مکانیکی) |
| `grid_frequency` | هرتز (Hz) | فرکانس لحظه‌ای شبکه برق متصل به نیروگاه |
| `power_output` | مگاوات (MW) | **متغیر هدف (Target)** - توان واقعی خروجی توربین |

* سیال عامل: ماده‌ای است که در نیروگاه انرژی را از یک بخش به بخش دیگر منتقل می‌کند؛ مثلاً در توربین بخار، بخار آب با فشار و دمای زیاد وارد توربین می‌شود، پره‌ها را می‌چرخاند و انرژی خود را به حرکت توربین و سپس تولید برق تبدیل می‌کند.

**💡 راهنمایی و مسیر تفکر:**

بر اساس دانش فیزیکی، رفتار این توربین خاص ترکیبی از اثرات خطی ساده و مکانیزم‌های غیرخطی پیچیده است. تیم مهندسی تأکید دارد که دو پدیده‌ی غیر خطی فیزیکی وجود دارد که یکی از آنها تابعی غیرخطی از یک ویژگی و دیگری تابعی غیرخطی از تفاضل دو ویژگی هست.
> ⚠️ کشف دقیق ریاضی این دو رابطه و تبدیل آن‌ها به ویژگی‌های جدید (Feature Engineering)، کلید تبدیل یک مدل خطی ساده به یک پیش‌بینی‌کننده‌ی قدرتمند است. داده‌ها با شما صحبت می‌کنند، کافی است نمودارهای درستی از آن‌ها رسم کنید.
</div>

<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">

## 3️⃣ چالش‌های کلیدی و مسیر تفکر

 در این بخش تعدادی از چالش های کلیدی مسئله مطرح شده است که در عملکرد مدل شما تاثیر خواهد گذاشت. البته چالش های بیشتری نیز ممکن است وجود داشته باشد.



### 🧹 ۱. مواجهه با داده‌های صنعتی خام

پایگاه‌داده‌های صنعتی هرگز تمیز نیستند. از فرمت‌های ترکیبی گرفته تا قطعی سنسورها، همگی بخشی از این داده‌ها هستند.


### ⚖️ ۲. اختلاف مقیاس متغیرها و نویز صنعتی

ویژگی‌های ورودی واحدهای اندازه‌گیری و دامنه تغییرات کاملاً متفاوتی دارند (مانند فشار، دما، و لرزش). همچنین داده‌های سنسوری و متغیر پاسخ ممکن است آلوده به نویزهای شدید یا مقادیر پرت (Outliers) باشند.


### 🔍 ۳. کشف بصری الگوهای غیرخطی

می‌دانیم که توان خروجی توربین با متغیرهای ورودی رابطه خطی ساده ندارد، بلکه تحت تأثیر اثرات منفرد هر متغیر و اثرات متقابل دوتایی میان آن‌ها قرار دارد.

</div>

<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">

### 📊 فرمول محاسبه نمره نهایی (سقف ۲۰ نمره)

نمره شما بر اساس دقت فایل `submission.csv` شما (بخش ۱۱) و به‌صورت یک عدد صحیح بین ۰ تا ۲۰ محاسبه می‌شود -- دقیقاً همان فرمولی که سرور داوری (`judge.sh`) اجرا می‌کند:

<div dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:embed">

$$
\text{Score} = \begin{cases}
0 & \text{RMSE} > 55.0 \\[4pt]
\operatorname{round}\big(5 + 15 \times \frac{55.0 - \text{RMSE}}{55.0 - 3.5}\big) & 3.5 < \text{RMSE} \le 55.0 \\[8pt]
20 & \text{RMSE} \le 3.5
\end{cases}
$$

</div>

- **نمره ۰ (Wrong Answer):** خطای مدل شما بیشتر از ۵۵.۰ (خطای Baseline) است -- عملکردی بدتر از یک رگرسیون خطی خامِ بدون مهندسی ویژگی.
- **نمره ۵ تا ۱۹ (Partially Correct، طیف پیوسته گردشده به عدد صحیح):** بین خطای Target و خطای Baseline؛ هرچه RMSE کمتر باشد نمره بالاتر می‌رود. **نمره‌ی ۲۰ در این بازه هرگز داده نمی‌شود** -- حتی اگر فرمول پیوسته به ۲۰ نزدیک شود، نمره در ۱۹ سقف می‌خورد؛ ۲۰ فقط برای رسیدن قطعی به هدف رزرو شده است.
- **نمره ۲۰ (Accepted / SOTA):** خطای مدل شما کمتر یا مساوی ۳.۵ است -- نیازمند کشف هر دو رابطه‌ی غیرخطی پنهان و پاک‌سازی دقیق داده‌هاست.

</div>

<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">

## 5️⃣ قالب تحویل (مهم -- پیش از شروع بخوانید)

**فقط یک فایل CSV تحویل می‌دهید -- نه نوت‌بوک، نه هیچ کد دیگری.** سرور داوری هیچ کدی از شما اجرا نمی‌کند؛ صرفاً فایل `submission.csv` شما را با برچسب‌های واقعی (محفوظ نزد سرور) مقایسه می‌کند.

**قوانین دقیق فایل تحویلی:**

- نام دقیق فایل: `submission.csv`
- دقیقاً دو ستون با همین نام‌ها: `id,power_output`
- برای هر سطر `test_queries.csv` باید دقیقاً یک سطر با همان مقدار `id` در `submission.csv` وجود داشته باشد (ترتیب سطرها اهمیتی ندارد -- تطبیق بر اساس `id` انجام می‌شود، نه بر اساس شماره‌ی سطر)
- بدون مقدار گمشده، NaN یا Inf در ستون `power_output`
- یک نمونه‌ی صحیح از نظر فرمت (با مقادیر ساختگی) در فایل `sample_submission.csv` کنار نوت‌بوک قرار دارد

بخش ۱۱ (انتهای نوت‌بوک) دقیقاً همین فایل را برای شما تولید می‌کند -- کافی است `train_model` و `predict_with_model` را کامل کرده باشید.

<div dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:embed">

```
id,power_output
1,412.7
2,298.1
...
```
</div>

⚠️ **قید مدل خطی، همچنان الزامی و بررسی‌شدنی است:** سرور به‌صورت خودکار کد شما را نمی‌بیند، اما پس از پایان مسابقه نوت‌بوک شما به‌صورت دستی بررسی می‌شود تا از خطی‌بودن مدل نهایی (بخش ۱) اطمینان حاصل شود. عدم رعایت این قید، تحویل شما را باطل می‌کند -- نوت‌بوک را دقیقاً همان‌طور که مدل نهایی را تولید کرده، نگه دارید و تغییر ندهید.

**کتابخانه‌های مجاز برای ساخت مدل:** `numpy`, `pandas`, `scipy`, `matplotlib`, `seaborn` و کتابخانه‌ی استاندارد پایتون.

</div>

<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">


## 6️⃣ بارگذاری داده‌ها و بررسی اولیه
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Labeled training data, and the unlabeled queries you must predict for.
train_df = pd.read_csv('train_data.csv')
test_queries_df = pd.read_csv('test_queries.csv')

print(f"Training data shape: {train_df.shape}")
print(f"Test queries shape: {test_queries_df.shape}")

# Inspect data types and initial properties
print("\n--- Initial info of training data ---")
train_df.info()
display(train_df.head())

print("\n--- Initial info of test queries ---")
test_queries_df.info()
display(test_queries_df.head())


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">


## 7️⃣ پیاده‌سازی مدل پایه از صفر

> ⚠️ **الزامی:** در این بخش باید الگوریتم رگرسیون خطی را **بدون استفاده از کتابخانه‌های آماده (نظیر sklearn)** و صرفاً با استفاده از عملیات ماتریسی **NumPy** پیاده‌سازی کنید تا کف نمره مسئله برای شما ثبت شود.
>
>
>

</div>

In [ ]:
class ScratchLinearRegression:
    """Simple linear regression implementation from scratch using NumPy"""
    def __init__(self):
        self.theta = None # This will include the bias term as theta[0]

    def fit(self, X, y):
        # ==============================================================================
        # TODO: 
        # 1. Add a column of ones to X for the bias term (intercept).
        # 2. Compute theta using the Normal Equation
        # Hint: Use np.linalg.pinv for numerical stability instead of np.linalg.inv
        # ==============================================================================
        pass

    def predict(self, X):
        # ==============================================================================
        # TODO: 
        # 1. Add a column of ones to X.
        # 2. Compute predictions
        # ==============================================================================
        pass

# You can test your scratch model here to establish a baseline


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">


## 8️⃣ فضای توسعه مدل اصلی

ساختار زیر برای سازمان‌دهی تفکر و کدهای شما پیشنهاد شده است.



### گام اول: پاک‌سازی و پیش‌پردازش داده‌ها

> 💡 **نکته‌ی مهم: داده‌ی آزمون هم کثیف است!**
> داده‌ی پنهان آزمون همان کثیفی‌های داده‌ی آموزش را دارد (فرمت رشته‌ای، مقادیر گمشده).
> بنابراین تابع پاک‌سازی شما باید در مرحله‌ی پیش‌بینی هم قابل استفاده باشد. اما یک دام بزرگ وجود دارد:
>
> ⚠️ **هرگز آماره‌هایی مثل میانه را از داده‌ی آزمون محاسبه نکنید!**
> چرا؟ چون مدل شما با توزیع داده‌ی آموزش یاد گرفته است. اگر در مرحله‌ی پیش‌بینی از میانه‌ی متفاوتی استفاده کنید، توزیع ورودی مدل تغییر می‌کند و پیش‌بینی‌ها خراب می‌شوند (این یک نوع Data Leakage است).
>
> ✅ **راه‌حل:** تابع خود را طوری طراحی کنید که:
> - در اولین صدا (مرحله‌ی آموزش): آماره‌ها را محاسبه و **برگرداند** تا ذخیره کنید.
> - در صدا‌های بعدی (مرحله‌ی پیش‌بینی): همان آماره‌ی ذخیره‌شده را **دریافت** و استفاده کند.

</div>

In [ ]:
# ==============================================================================
# 1. Data Cleaning & Preprocessing
# ==============================================================================

def preprocess_data(df, stats=None):
    """
    Input: Raw DataFrame, optional stats dict
    Output: Cleaned DataFrame, stats dict
    
    Hint: در اولین بار (fit)، stats=None بگذارید تا میانه‌ها محاسبه شوند.
          در دفعات بعدی (predict)، stats ذخیره‌شده را پاس دهید.
    """
    df_clean = df.copy()

    # parse numeric columns (HINT: some columns might contain strings like "45°C" or "50 Hz". You must extract the numeric part.)
    # handle missing values (HINT: simple mean/median imputation might work).
    # but be careful with outliers

    #TODO:

    return df_clean, stats

# train_clean = preprocess_data(train_df)


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">

### گام دوم: اکتشاف بصری (Visual EDA) جهت مهندسی ویژگی

در این بخش، کدهایی بنویسید که به شما کمک کند روابط پنهان در داده‌ها را **ببینید**.

</div>

In [ ]:
# ==============================================================================
# 2. Visual EDA for Feature Discovery (Guided)
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_single_feature_with_trend(df, feature_col, target_col='power_output'):
    """
    رسم نمودار پراکندگی همراه با خط روند (Lowess) برای کشف الگوهای غیرخطی تک‌متغیره.
    خط روند به شما کمک می‌کند تا شکل کلی رابطه را در میان نویز داده‌ها ببینید.
    """
    plt.figure(figsize=(8, 5))
    sns.regplot(x=feature_col, y=target_col, data=df, scatter_kws={'alpha':0.3}, lowess=True, line_kws={'color':'red', 'linewidth':2})
    plt.title(f'Relationship between {feature_col} and {target_col}')
    plt.xlabel(feature_col)
    plt.ylabel(target_col)
    plt.grid(True, alpha=0.3)
    plt.show()

def explore_interaction_hypothesis(df, feature1, feature2, target_col='power_output'):
    """
    بررسی فرضیه‌ی اثر متقابل یا ترکیبی دو متغیر.
    گاهی اوقات رابطه‌ی اصلی با یک متغیر جدید که ترکیبی از دو متغیر موجود است، آشکار می‌شود.
    """
    df_temp = df.copy()
    df_temp['combined_feature'] = df_temp[feature1] - df_temp[feature2]
    
    plt.figure(figsize=(8, 5))
    sns.regplot(x='combined_feature', y=target_col, data=df_temp, scatter_kws={'alpha':0.3}, lowess=True, line_kws={'color':'red', 'linewidth':2})
    plt.title(f'Relationship between ({feature1} - {feature2}) and {target_col}')
    plt.xlabel(f'{feature1} - {feature2}')
    plt.ylabel(target_col)
    plt.grid(True, alpha=0.3)
    plt.show()

# ==============================================================================
# 🎯 چالش اکتشاف
# ==============================================================================
# اکنون که ابزار رسم نمودار را دارید، خودتان با نوشتن حلقه‌ به جستجو بپردازید:
# 
# ۱. تک‌متغیره: یک حلقه بنویسید که تابع اول را برای تمام ستون‌های عددی اجرا کند 
#    تا رابطه‌ی غیرخطی تک‌متغیره را شناسایی کنید.
# 
# ۲. دو‌متغیره: می‌دانیم رابطه‌ی غیرخطی دوم، تابعی از «اختلاف» دو ویژگی است. 
#    با نوشتن حلقه‌ای روی تمام جفت‌های ممکن فیچرها و استفاده از تابع دوم، نمودار ها را رسم کنید تا الگوی دقیق را کشف نمایید.
#
# هدف نهایی: پیدا کردن دقیق نام آن ویژگی‌ها و تبدیل ریاضی آن‌هاست که خط روند را 
# به یک رابطه‌ی خطیِ واضح تبدیل می‌کند.

<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">


### گام سوم:  ساخت ویژگی و آموزش مدل نهایی

بر اساس مشاهدات بصری خود، ویژگی‌های جدید را اضافه کنید.

</div>

In [ ]:
# ==============================================================================
# 3. Feature Expansion & Local Validation Pipeline
# ==============================================================================

def generate_discovered_features(X):
    """
    Generate new features strictly based on the insights gained from the Visual EDA.
    Add the necessary single-feature transformations and pairwise interactions here.
    """
    X_expanded = X.copy()

    # TODO: create specific transformed features based on your visual findings
    # TODO: create specific interaction features based on your visual findings

    return X_expanded

# X_train_expanded = generate_discovered_features(X_train_clean)


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">


## 9️⃣ آموزش مدل نهایی

توابعی که در بخش ۸ نوشتید (`preprocess_data`, `generate_discovered_features`) را اینجا در دو تابع ساده‌ی `train_model` و `predict_with_model` جمع‌بندی کنید.

> 💡 **نکته‌ی مهم: یکپارچگی train و predict**
> آماره‌هایی که در `train_model` محاسبه می‌کنید (مثل میانه‌ها برای پر کردن مقادیر گمشده، یا وزن‌های مدل)
> باید داخل دیکشنری خروجی `model` قرار بگیرند و در `predict_with_model` دقیقاً از **همان مقادیر ذخیره‌شده**
> استفاده شود.
>
> ⚠️ هرگز داخل `predict_with_model` آماره‌ی جدید از روی داده‌ی ورودی محاسبه نکنید؛ این کار باعث ناسازگاری
> بین آموزش و پیش‌بینی و افت شدید دقت می‌شود.

</div>

In [ ]:
def train_model(train_df):
    """
    Train your final model on the full labeled training DataFrame.
    Return a single `model` dict containing everything predict_with_model will
    need later: cleaning stats, feature-engineering parameters, and the fitted
    regression weights. Nothing computed here may depend on test_queries.csv.

    Reuse preprocess_data / generate_discovered_features / ScratchLinearRegression
    (or sklearn's linear models) from the sections above.
    """
    # TODO: df_clean, stats = preprocess_data(train_df, stats=None)
    # TODO: X_expanded = generate_discovered_features(...)
    # TODO: fit your final linear model on (X_expanded, y)
    model = {}
    return model


def predict_with_model(model, query_df):
    """
    Apply the EXACT SAME preprocessing + feature expansion used inside
    train_model (using the stats/params stored in `model`, never recomputed
    from query_df), then return one prediction per row of query_df, in the
    same row order as query_df.
    """
    # TODO: df_clean, _ = preprocess_data(query_df, stats=model['stats'])
    # TODO: X_expanded = generate_discovered_features(df_clean)
    # TODO: preds = ... (use model['weights'] / model['final_model'])
    preds = None
    return preds


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">


## 🔟 ارزیابی محلی (Local Validation)

این سلول بخشی از `train_data.csv` را کنار می‌گذارد، `train_model` را روی باقیمانده اجرا می‌کند، و روی بخش کنارگذاشته‌شده با `predict_with_model` پیش‌بینی می‌گیرد -- تا پیش از تولید فایل نهایی، تخمینی از عملکرد مدل خود به‌دست آورید.

⚠️ توجه: این فقط یک **تخمین محلی** است. سرور داوری کدی از شما اجرا نمی‌کند؛ نمره‌ی نهایی شما از مقایسه‌ی `submission.csv` (بخش ۱۱) با داده‌ی آزمون واقعی محاسبه می‌شود.

</div>

In [ ]:
def run_local_validation(full_train_df, seed=42, holdout_frac=0.2):
    """Estimates your score locally by holding out part of train_data.csv
    (no separate labeled test file exists anywhere)."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(full_train_df))
    n_holdout = int(len(full_train_df) * holdout_frac)
    holdout_idx, fit_idx = idx[:n_holdout], idx[n_holdout:]

    fit_df = full_train_df.iloc[fit_idx].reset_index(drop=True)
    holdout_df = full_train_df.iloc[holdout_idx].reset_index(drop=True)
    queries_df = holdout_df.drop(columns=['power_output'])
    true_y = holdout_df['power_output'].values

    model = train_model(fit_df)
    preds = np.asarray(predict_with_model(model, queries_df), dtype=float)

    assert len(preds) == len(queries_df), "predict_with_model must return one value per query row"
    assert not np.isnan(preds).any(), "predict_with_model returned NaN"

    rmse = float(np.sqrt(np.mean((true_y - preds) ** 2)))
    print(f"Estimated held-out RMSE: {rmse:.3f}")

    rmse_baseline = 55.0
    rmse_target = 3.5

    # Mirrors judge.sh exactly: integer score, clamped to [5, 19] in the
    # partial-credit band so it can never round up to 20 before RMSE
    # actually reaches the target.
    if rmse > rmse_baseline:
        score = 0
    elif rmse <= rmse_target:
        score = 20
    else:
        raw = 5 + 15 * ((rmse_baseline - rmse) / (rmse_baseline - rmse_target))
        score = max(5, min(19, round(raw)))

    print(f"Estimated Score: {score} / 20 (Baseline RMSE ~ {rmse_baseline}, Target RMSE ~ {rmse_target})")
    return rmse

# run_local_validation(train_df)


<div dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:embed">


## 1️⃣1️⃣ ساخت فایل تحویل (`submission.csv`)

این سلول، مدل نهایی شما را روی **تمام** `train_data.csv` آموزش می‌دهد، روی `test_queries.csv` (داده‌ی آزمون واقعی که در بخش ۶ بارگذاری کردید) پیش‌بینی می‌گیرد، و فایل `submission.csv` را دقیقاً با فرمت مورد نیاز (بخش ۵) می‌سازد.

**این تنها فایلی است که در پایان آپلود می‌کنید.**

</div>

In [ ]:
# ==============================================================================
# Final submission: train on ALL of train_data.csv, predict on test_queries.csv
# ==============================================================================

final_model = train_model(train_df)

query_ids = test_queries_df['id'].values
final_preds = np.asarray(
    predict_with_model(final_model, test_queries_df.drop(columns=['id'])),
    dtype=float,
)

assert len(final_preds) == len(test_queries_df), "one prediction per row of test_queries.csv is required"
assert not np.isnan(final_preds).any() and not np.isinf(final_preds).any(), "predictions contain NaN/Inf"

submission_df = pd.DataFrame({'id': query_ids, 'power_output': final_preds})
submission_df.to_csv('submission.csv', index=False)

print(f"Wrote submission.csv with {len(submission_df)} rows.")
display(submission_df.head())
